In [ ]:
#cài đặt thư viện
!pip install -q lightning
#https://lightning.ai/docs/pytorch/stable/

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 17.3 MB/s eta 0:00:00


In [ ]:
#download dataset
#!wget https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv
!gdown 1aXs9anuFLEOO2iQ9mUYDOkni04v_Fy9J

Downloading...
From: https://drive.google.com/uc?id=1aXs9anuFLEOO2iQ9mUYDOkni04v_Fy9J
To: /content/iris.csv
100% 3.98k/3.98k [00:00<00:00, 15.5MB/s]


In [ ]:
import lightning as L
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
#read dataset
data = pd.read_csv("/content/iris.csv")
data.head()

,sepal.length,sepal.width,petal.length,petal.width,variety
0,5.1,3.5,1.4,0.2,Setosa
1,4.9,3.0,1.4,0.2,Setosa
2,4.7,3.2,1.3,0.2,Setosa
3,4.6,3.1,1.5,0.2,Setosa
4,5.0,3.6,1.4,0.2,Setosa


Iris là dataset nhận dạng loại hoa từ 4 features bao gồm sepal.length, sepal.widt, petal.length, và petal.width

In [ ]:
#encode cho label
labels = {}
for index, element in enumerate(data["variety"].unique()):
  labels[element] = index

data.loc[:, "labels"] = data["variety"].apply(lambda x: labels[x])
data.head()

,sepal.length,sepal.width,petal.length,petal.width,variety,labels
0,5.1,3.5,1.4,0.2,Setosa,0
1,4.9,3.0,1.4,0.2,Setosa,0
2,4.7,3.2,1.3,0.2,Setosa,0
3,4.6,3.1,1.5,0.2,Setosa,0
4,5.0,3.6,1.4,0.2,Setosa,0


In [ ]:
# Tạo class Dataset để đọc dữ liệu
COLUMNS = data.drop(['variety', "labels"], axis=1).columns.to_list()  # Lấy danh sách các cột không bao gồm 'variety' và 'labels'

class DataSet(torch.utils.data.Dataset):
    def __init__(self, data, normalizer, columns=COLUMNS):
        super(DataSet, self).__init__()  # Khởi tạo lớp cha
        self.data = data  # Lưu trữ dữ liệu đầu vào
        self.feature = normalizer.transform(self.data[columns].values)  # Chuẩn hóa các đặc trưng
        self.feature = torch.tensor(self.feature).float()  # Chuyển đổi các đặc trưng thành tensor kiểu float
        self.label = torch.tensor(self.data["labels"].values)  # Chuyển đổi nhãn thành tensor

    def __len__(self):
        return len(self.feature)  # Trả về số lượng mẫu trong dataset

    def __getitem__(self, idx):
        # Trả về một từ điển chứa đặc trưng và nhãn tại chỉ số idx
        return {"feature": self.feature[idx],
                "label": self.label[idx]}


In [ ]:
# Chia và chuẩn hóa dataset
from sklearn.preprocessing import StandardScaler  # Nhập StandardScaler từ thư viện sklearn

BATCH_SIZE = 32  # Kích thước batch cho DataLoader
# Chia dữ liệu thành tập huấn luyện và tập kiểm tra (80% - 20%)
train_data, test_data = train_test_split(data, test_size=0.2, random_state=0)
# Chia tập huấn luyện thành tập kiểm tra và tập val (60% train, 20% test 20% val)
test_data, val_data = train_test_split(train_data, test_size=0.25, random_state=0)

# Khởi tạo StandardScaler để chuẩn hóa dữ liệu
normalizer = StandardScaler()
# Tính toán các tham số chuẩn hóa dựa trên tập huấn luyện
normalizer.fit(train_data[COLUMNS].values)

StandardScaler()

In [ ]:
# Tạo DataLoader cho tập huấn luyện, cho phép trộn và sử dụng nhiều workers
train_loader = torch.utils.data.DataLoader(DataSet(train_data, normalizer), batch_size=BATCH_SIZE,
                                           shuffle=True, num_workers=2)

# Tạo DataLoader cho tập val
val_loader = torch.utils.data.DataLoader(DataSet(val_data, normalizer), batch_size=BATCH_SIZE, num_workers=2)

# Tạo DataLoader cho tập kiểm tra
test_loader = torch.utils.data.DataLoader(DataSet(test_data, normalizer), batch_size=BATCH_SIZE, num_workers=2)

# Lấy một batch dữ liệu từ train_loader
data_loader = next(iter(train_loader))
# In ra kích thước của các đặc trưng trong batch
print(len(data_loader["feature"]))
# In ra kích thước của các nhãn trong batch
print(len(data_loader["label"]))
# In ra các đặc trưng và nhãn (bình luận lại để không in ra)
#print(data_loader["feature"], "\n", data_loader["label"])


32
32


In [ ]:
len(COLUMNS)

4

In [ ]:
# Tạo model
class Model(L.LightningModule):
    def __init__(self, num_classes=len(labels), learning_rate=5e-3, input_dim=len(COLUMNS)):
        super(Model, self).__init__()  # Khởi tạo lớp cha
        self.learning_rate = learning_rate  # Lưu trữ learning rate
        # Tạo mô hình MLP với các lớp tuyến tính và hàm kích hoạt ReLU
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(input_dim, 10),  # Lớp đầu vào với kích thước input_dim và đầu ra 64
            torch.nn.ReLU(),                  # Hàm kích hoạt ReLU
            torch.nn.Linear(10, num_classes), # Lớp đầu ra với đầu vào 32 và đầu ra số lớp
        )

    def forward(self, x):
        return self.mlp(x)  # Thực hiện phép biến đổi thông qua mô hình MLP

    def training_step(self, batch, batch_idx):
        x = batch["feature"]  # Lấy đặc trưng từ batch
        y = batch["label"]    # Lấy nhãn từ batch
        y_pred = self(x)      # Dự đoán nhãn bằng mô hình
        # Tính toán loss bằng hàm CrossEntropy
        loss = torch.nn.CrossEntropyLoss()(y_pred, y)
        self.log('train_loss', loss, prog_bar=True)  # Ghi lại loss huấn luyện để hiển thị
        return loss  # Trả về loss

    def validation_step(self, batch, batch_idx):
        x = batch["feature"]  # Lấy đặc trưng từ batch
        y = batch["label"]    # Lấy nhãn từ batch
        y_pred = self(x)      # Dự đoán nhãn bằng mô hình
        # Tính toán loss cho tập val
        val_loss = torch.nn.CrossEntropyLoss()(y_pred, y)
        self.log('val_loss', val_loss, prog_bar=True)  # Ghi lại loss val để hiển thị
        return val_loss  # Trả về loss val

    def configure_optimizers(self):
        # Khởi tạo optimizer Adam với learning rate đã định
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        return optimizer  # Trả về optimizer

    @torch.no_grad()
    def evaluate(self, dataloader):
        self.eval()

        y_true = []
        y_pred = []

        device = self.device

        for batch in dataloader:
            x = batch["feature"].to(device)
            y = batch["label"].to(device)

            logits = self(x)
            preds = torch.argmax(logits, dim=1)

            y_true.extend(y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

        acc = accuracy_score(y_true, y_pred)

        return {
            "accuracy": acc,
        }


In [ ]:
# Train model
model = Model()  # Khởi tạo một thể hiện của mô hình

# Tạo Trainer từ PyTorch Lightning với số epoch tối đa là 100 và kích hoạt tính năng phát hiện bất thường
trainer = L.Trainer(max_epochs=50)

# Bắt đầu quá trình huấn luyện mô hình với dữ liệu huấn luyện và val
trainer.fit(model, train_loader, val_loader)


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

┏━━━┳━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ mlp  │ Sequential │     83 │ train │     0 │
└───┴──────┴────────────┴────────┴───────┴───────┘

Trainable params: 83                                                                                               
Non-trainable params: 0                                                                                            
Total params: 83                                                                                                   
Total estimated model params size (MB): 0.000                                                                      
Modules in train mode: 4                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/loops/fit_loop.py:321: The number of training batches (4)
is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you 
want to see logs for the training epoch.

INFO: `Trainer.fit` stopped: `max_epochs=50` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.


In [ ]:
model.evaluate(test_loader)

{'accuracy': 0.9444444444444444}

In [ ]:
model.evaluate(train_loader)

{'accuracy': 0.925}

#Bài tập

Hãy thay đổi cấu trúc mô hình MLP hiện tại bằng cách sử dụng 2 lớp Dense (Fully Connected) với số lượng nút ẩn lần lượt là 32 và 16. Giữ nguyên lớp đầu ra và các hàm kích hoạt ReLU cho các lớp ẩn. Sau đó huấn luyện lại mô hình và so sánh các chỉ số đánh giá (Accuracy, Precision, Recall, F1-score) với mô hình ban đầu.

Train với batch size 16, epoch 50, in kết quả các metrics Precsion, Recall, F1 Score và nhận xét

